In [1]:
import pandas as pd
arquivo = "/content/Demonstrativo.csv"

df=pd.read_csv("Demonstrativo.csv",sep=";",header=None, encoding='latin1')
df.columns=["ANO","CENARIO","CONTA","VALOR"]
df.head(10)


,ANO,CENARIO,CONTA,VALOR
0,Ano 1,Total Cen_00001,NaN,"0,025138815"
1,Ano 1,Total Cen_00001,BAL - Total do Ativo,"10.631.127.075,2579"
2,Ano 1,Total Cen_00001,BAL - Ativo Circulante,"1.795.720.872,17537"
3,Ano 1,Total Cen_00001,BAL - DisponÃ­vel,"1.205.264.965,51818"
4,Ano 1,Total Cen_00001,BAL - Contas a Receber - SWAP,"95.680.421,9232409"
5,Ano 1,Total Cen_00001,BAL - Contas a Receber - Partes Relacionadas,"584.661,46688296"
6,Ano 1,Total Cen_00001,BAL - Contas a Receber - Clientes,"453.870.957,151068"
7,Ano 1,Total Cen_00001,BAL - Estoques Diversos,"86.620,78872148"
8,Ano 1,Total Cen_00001,BAL - Outros CrÃ©ditos,"40.233.245,3272782"
9,Ano 1,Total Cen_00001,BAL - RealizÃ¡vel a Longo Prazo,"639.023.858,22772"


In [8]:
import pandas as pd
from IPython.display import display, HTML


# 1. SELEÇÃO DOS DADOS DA DRE


dre = df[df["CONTA"].str.startswith("DRE", na=False)].copy()


# 2. TRATAMENTO DOS VALORES


dre["NOVOVALOR"] = (dre["VALOR"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False))

dre["NOVOVALOR"] = pd.to_numeric(dre["NOVOVALOR"],errors="coerce")


# 3. CONVERSÃO PARA BILHÕES


dre["VALOR_BILHOES"] = (dre["NOVOVALOR"] / 1_000_000_000)


# 4. TRATAMENTO DO ANO


dre["NUM_ANO"] = (dre["ANO"].astype(str).str.extract(r"(\d+)")[0])

dre["NUM_ANO"] = pd.to_numeric(dre["NUM_ANO"],errors="coerce")


# 5. LIMPEZA DOS NOMES DAS CONTAS


dre["CONTA_LIMPA"] = (dre["CONTA"].astype(str).str.replace(r"^DRE\s*-\s*","",regex=True).str.strip())


# 6. SOMA DE TODOS OS CENÁRIOS


dre_consolidada = (dre.groupby(["CONTA_LIMPA", "NUM_ANO"],as_index=False)["VALOR_BILHOES"].sum())


# 7. ORDEM ORIGINAL DAS CONTAS


ordem_contas_dre = (dre["CONTA_LIMPA"].dropna().drop_duplicates().tolist())


# 8. TABELA DINÂMICA


tabela_dre = dre_consolidada.pivot_table(index="CONTA_LIMPA",columns="NUM_ANO",values="VALOR_BILHOES",aggfunc="sum",fill_value=0)


# Mantém exatamente a ordem original
tabela_dre = tabela_dre.reindex(
    [
        conta
        for conta in ordem_contas_dre
        if conta in tabela_dre.index
    ]
)


# Ordena os anos
tabela_dre = tabela_dre.reindex(
    sorted(tabela_dre.columns),
    axis=1
)


# 9. ANOS


anos_dre = [int(ano)
    for ano in tabela_dre.columns
]


# 10. CORES PADRÃO


# Mesmo padrão visual do BAL

COR_BARRA_PRINCIPAL = "#4472C4"

COR_VERDE = "#70AD47"

COR_VERDE_CLARO = "#C6E0B4"

COR_FUNDO = "#F7FBF4"

COR_TOTAL = "#17365D"

COR_TOTAL_BORDA = "#0E243F"

COR_TEXTO = "#1F2937"

COR_BRANCO = "#FFFFFF"

COR_BORDA = "#D9E2F3"


# 11. FORMATAÇÃO DOS VALORES


def formatar_valor_dre(valor):

    if pd.isna(valor):
        return "R$ -"

    if abs(valor) < 0.005:
        return "R$ -"

    numero = (
        f"{abs(valor):,.2f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    if valor < 0:
        return f"- R$ {numero}"

    return f"R$ {numero}"


# 12. IDENTIFICAÇÃO AUTOMÁTICA DAS LINHAS IMPORTANTES


# Resultado final da DRE
linhas_resultado_final = [
    conta
    for conta in tabela_dre.index
    if any(
        palavra in conta.lower()
        for palavra in [
            "resultado líquido",
            "resultado liquido",
            "lucro líquido",
            "lucro liquido"
        ]
    )
]


# Resultados / subtotais importantes
linhas_subtotais = [
    conta
    for conta in tabela_dre.index
    if any(
        palavra in conta.lower()
        for palavra in [
            "ebitda",
            "ebit",
            "lucro bruto",
            "resultado bruto",
            "resultado operacional",
            "resultado financeiro",
            "resultado antes"
        ]
    )
    and conta not in linhas_resultado_final
]


# Receita principal
linhas_receita = [
    conta
    for conta in tabela_dre.index
    if conta.lower().strip() in [
        "receita",
        "receita líquida",
        "receita liquida",
        "receita operacional",
        "receita operacional líquida",
        "receita operacional liquida"
    ]
]


# 13. CABEÇALHO DOS ANOS


cabecalho_anos = ""

for ano in anos_dre:

    cabecalho_anos += f"""
    <th style="
        background:{COR_VERDE};
        color:{COR_BRANCO};
        padding:9px 14px;
        text-align:center;
        border:1px solid {COR_BORDA};
        min-width:110px;
    ">
        Ano {ano}
    </th>
    """


# 14. FUNÇÃO PARA CRIAR LINHA


def criar_linha_dre(
    conta,
    fundo=COR_FUNDO,
    cor=COR_TEXTO,
    negrito=False,
    borda_superior=f"1px solid {COR_BORDA}",
    borda_inferior=f"1px solid {COR_BORDA}"
):

    if conta not in tabela_dre.index:
        return ""

    peso = "bold" if negrito else "normal"

    html = f"""
    <tr>

        <td style="
            background:{fundo};
            color:{cor};
            font-weight:{peso};
            padding:9px 12px;
            border-left:1px solid {COR_BORDA};
            border-right:1px solid {COR_BORDA};
            border-top:{borda_superior};
            border-bottom:{borda_inferior};
            white-space:nowrap;
            min-width:320px;
        ">
            {conta}
        </td>
    """

    for ano in anos_dre:

        valor = tabela_dre.loc[
            conta,
            ano
        ]

        html += f"""
        <td style="
            background:{fundo};
            color:{cor};
            font-weight:{peso};
            padding:9px 12px;
            text-align:right;
            border-left:1px solid {COR_BORDA};
            border-right:1px solid {COR_BORDA};
            border-top:{borda_superior};
            border-bottom:{borda_inferior};
            white-space:nowrap;
        ">
            {formatar_valor_dre(valor)}
        </td>
        """

    html += "</tr>"

    return html


# 15. FUNÇÃO PARA CRIAR BARRA PRINCIPAL


def criar_barra_principal_dre(titulo):

    return f"""
    <tr>

        <td
            colspan="{len(anos_dre) + 1}"

            style="
                background:{COR_BARRA_PRINCIPAL};
                color:{COR_BRANCO};
                font-weight:bold;
                font-size:14px;
                padding:10px 12px;
                border:1px solid {COR_BARRA_PRINCIPAL};
                text-align:left;
            "
        >
            {titulo}
        </td>

    </tr>
    """


# 16. CONSTRUÇÃO DAS LINHAS


linhas_html = ""


# BARRA PRINCIPAL AZUL


linhas_html += criar_barra_principal_dre(
    "DEMONSTRAÇÃO DO RESULTADO"
)


# CONTAS DA DRE


for conta in tabela_dre.index:


    # --------------------------------------------------------
    # RESULTADO LÍQUIDO
    # Azul-escuro igual aos totais do BAL
    # --------------------------------------------------------

    if conta in linhas_resultado_final:

        linhas_html += criar_linha_dre(
            conta,
            fundo=COR_TOTAL,
            cor=COR_BRANCO,
            negrito=True,
            borda_superior=f"3px solid {COR_TOTAL_BORDA}",
            borda_inferior=f"3px solid {COR_TOTAL_BORDA}"
        )


    # --------------------------------------------------------
    # EBITDA / EBIT / OUTROS RESULTADOS
    # Verde
    # --------------------------------------------------------

    elif conta in linhas_subtotais:

        linhas_html += criar_linha_dre(
            conta,
            fundo=COR_VERDE,
            cor=COR_BRANCO,
            negrito=True,
            borda_superior="2px solid #548235",
            borda_inferior="2px solid #548235"
        )


    # --------------------------------------------------------
    # RECEITA
    # Verde-claro
    # --------------------------------------------------------

    elif conta in linhas_receita:

        linhas_html += criar_linha_dre(
            conta,
            fundo=COR_VERDE_CLARO,
            cor=COR_TEXTO,
            negrito=True
        )


    # --------------------------------------------------------
    # DEMAIS CONTAS
    # --------------------------------------------------------

    else:

        linhas_html += criar_linha_dre(
            conta,
            fundo=COR_FUNDO,
            cor=COR_TEXTO
        )


# 17. TABELA COMPLETA


html_dre = f"""

<div style="
    width:100%;
    overflow-x:auto;
    font-family:Arial, sans-serif;
    margin-top:20px;
    margin-bottom:35px;
">

    <table style="
        border-collapse:collapse;
        width:max-content;
        min-width:100%;
        font-size:13px;
    ">

        <thead>


            <!-- ==================================================
                 TÍTULO PRINCIPAL
                 ================================================== -->

            <tr>

                <th style="
                    background:{COR_TOTAL};
                    color:{COR_BRANCO};
                    padding:11px;
                    text-align:center;
                    border:1px solid {COR_BORDA};
                ">
                    DRE
                </th>


                <th
                    colspan="{len(anos_dre)}"

                    style="
                        background:{COR_TOTAL};
                        color:{COR_BRANCO};
                        padding:11px;
                        text-align:center;
                        border:1px solid {COR_BORDA};
                        font-size:15px;
                    "
                >

                    Demonstração do Resultado do Exercício
                    — Valores em R$ bilhões

                </th>

            </tr>


            <!-- ==================================================
                 CABEÇALHO DOS ANOS
                 ================================================== -->

            <tr>

                <th style="
                    background:{COR_VERDE};
                    color:{COR_BRANCO};
                    padding:9px 12px;
                    text-align:left;
                    border:1px solid {COR_BORDA};
                    min-width:320px;
                ">

                    CONTA

                </th>

                {cabecalho_anos}

            </tr>

        </thead>


        <tbody>

            {linhas_html}

        </tbody>


    </table>

</div>

"""


# 18. EXIBIÇÃO


display(HTML(html_dre))

In [9]:
import pandas as pd
from IPython.display import display, HTML


# 1. SELEÇÃO DOS DADOS DO BAL


bal = df[df["CONTA"].str.startswith("BAL", na=False)].copy()


# 2. TRATAMENTO DOS VALORES


bal["NOVOVALOR"] = (bal["VALOR"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False))

bal["NOVOVALOR"] = pd.to_numeric(bal["NOVOVALOR"],errors="coerce")


# 3. CONVERSÃO PARA BILHÕES


bal["VALOR_BILHOES"] = (bal["NOVOVALOR"] / 1_000_000_000)


# 4. TRATAMENTO DO ANO


bal["NUM_ANO"] = (bal["ANO"].astype(str).str.extract(r"(\d+)")[0])

bal["NUM_ANO"] = pd.to_numeric(bal["NUM_ANO"],errors="coerce")


# 5. LIMPEZA DOS NOMES DAS CONTAS


bal["CONTA_LIMPA"] = (bal["CONTA"].astype(str).str.replace(r"^BAL\s*-\s*","",regex=True).str.strip())


# 6. SOMA DE TODOS OS CENÁRIOS


bal_consolidado = (bal.groupby(["CONTA_LIMPA", "NUM_ANO"],as_index=False)["VALOR_BILHOES"].sum())


# 7. TABELA DINÂMICA


tabela_bal = bal_consolidado.pivot_table(index="CONTA_LIMPA",columns="NUM_ANO",values="VALOR_BILHOES",aggfunc="sum",fill_value=0)

tabela_bal = tabela_bal.reindex(sorted(tabela_bal.columns),axis=1)


# 8. ORDEM ORIGINAL DAS CONTAS

ordem_original = (bal["CONTA_LIMPA"].dropna().drop_duplicates().tolist())


# 9. FUNÇÕES PARA LOCALIZAR CONTAS


def encontrar_conta(opcoes):

    for opcao in opcoes:

        if opcao in ordem_original:
            return opcao

    return None


def contas_entre(inicio, finais):

    if inicio is None:
        return []

    if inicio not in ordem_original:
        return []

    indice_inicio = ordem_original.index(inicio)
    indice_fim = len(ordem_original)

    for final in finais:

        if final is not None and final in ordem_original:

            posicao = ordem_original.index(final)

            if (
                posicao > indice_inicio
                and posicao < indice_fim
            ):
                indice_fim = posicao

    return ordem_original[
        indice_inicio:indice_fim
    ]


# 10. IDENTIFICAÇÃO DAS CONTAS PRINCIPAIS


conta_ativo_circulante = encontrar_conta([
    "Ativo Circulante"
])

conta_ativo_nao_circulante = encontrar_conta([
    "Ativo Não Circulante",
    "Realizável a Longo Prazo"
])

conta_total_ativo = encontrar_conta([
    "Total do Ativo"
])


conta_passivo_circulante = encontrar_conta([
    "Passivo Circulante"
])

conta_passivo_nao_circulante = encontrar_conta([
    "Passivo Não Circulante",
    "Exigível a Longo Prazo"
])

conta_total_passivo = encontrar_conta([
    "Total do Passivo"
])


conta_patrimonio = encontrar_conta([
    "Patrimônio Líquido"
])

conta_total_patrimonio = encontrar_conta([
    "Total do Patrimônio Líquido"
])


# 11. ATIVO CIRCULANTE


ativo_circulante = contas_entre(
    conta_ativo_circulante,
    [
        conta_ativo_nao_circulante,
        conta_total_ativo,
        conta_passivo_circulante
    ]
)


# 12. ATIVO NÃO CIRCULANTE


ativo_nao_circulante = contas_entre(
    conta_ativo_nao_circulante,
    [
        conta_total_ativo,
        conta_passivo_circulante
    ]
)


# 13. PASSIVO CIRCULANTE


passivo_circulante = contas_entre(
    conta_passivo_circulante,
    [
        conta_passivo_nao_circulante,
        conta_patrimonio,
        conta_total_passivo
    ]
)


# 14. PASSIVO NÃO CIRCULANTE


passivo_nao_circulante = contas_entre(
    conta_passivo_nao_circulante,
    [
        conta_patrimonio,
        conta_total_passivo
    ]
)


# 15. PATRIMÔNIO LÍQUIDO


patrimonio_liquido = contas_entre(
    conta_patrimonio,
    [
        conta_total_patrimonio,
        conta_total_passivo
    ]
)


patrimonio_detalhes = [
    conta
    for conta in patrimonio_liquido
    if conta not in [
        conta_patrimonio,
        conta_total_patrimonio,
        conta_total_passivo
    ]
]


# 16. TOTAL DO PATRIMÔNIO LÍQUIDO


if conta_total_patrimonio is not None:

    conta_usada_total_patrimonio = (
        conta_total_patrimonio
    )

else:

    conta_usada_total_patrimonio = (
        conta_patrimonio
    )


# 17. ANOS


anos_bal = [
    int(ano)
    for ano in tabela_bal.columns
]


# 18. CORES PADRÃO


COR_BARRA_PRINCIPAL = "#4472C4"
COR_VERDE = "#70AD47"
COR_VERDE_CLARO = "#C6E0B4"
COR_FUNDO = "#F7FBF4"
COR_TOTAL = "#17365D"
COR_TOTAL_BORDA = "#0E243F"
COR_TEXTO = "#1F2937"
COR_BRANCO = "#FFFFFF"
COR_BORDA = "#D9E2F3"


# 19. FORMATAÇÃO DOS VALORES


def formatar_valor_bal(valor):

    if pd.isna(valor):
        return "R$ -"

    if abs(valor) < 0.005:
        return "R$ -"

    numero = (
        f"{abs(valor):,.2f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    if valor < 0:
        return f"- R$ {numero}"

    return f"R$ {numero}"


# 20. CABEÇALHO DOS ANOS


cabecalho_anos = ""

for ano in anos_bal:

    cabecalho_anos += f"""
    <th style="
        background:{COR_VERDE};
        color:{COR_BRANCO};
        padding:9px 14px;
        text-align:center;
        border:1px solid {COR_BORDA};
        min-width:110px;
    ">
        Ano {ano}
    </th>
    """


# 21. FUNÇÃO PARA CRIAR LINHAS


def criar_linha(
    conta,
    nome_exibido=None,
    fundo=COR_BRANCO,
    cor=COR_TEXTO,
    negrito=False,
    borda_superior=f"1px solid {COR_BORDA}",
    borda_inferior=f"1px solid {COR_BORDA}"
):

    if conta is None:
        return ""

    if conta not in tabela_bal.index:
        return ""

    if nome_exibido is None:
        nome_exibido = conta

    peso = "bold" if negrito else "normal"

    html = f"""
    <tr>

        <td style="
            background:{fundo};
            color:{cor};
            font-weight:{peso};
            padding:9px 12px;
            border-left:1px solid {COR_BORDA};
            border-right:1px solid {COR_BORDA};
            border-top:{borda_superior};
            border-bottom:{borda_inferior};
            white-space:nowrap;
            min-width:320px;
        ">
            {nome_exibido}
        </td>
    """

    for ano in anos_bal:

        valor = tabela_bal.loc[
            conta,
            ano
        ]

        html += f"""
        <td style="
            background:{fundo};
            color:{cor};
            font-weight:{peso};
            padding:9px 12px;
            text-align:right;
            border-left:1px solid {COR_BORDA};
            border-right:1px solid {COR_BORDA};
            border-top:{borda_superior};
            border-bottom:{borda_inferior};
            white-space:nowrap;
        ">
            {formatar_valor_bal(valor)}
        </td>
        """

    html += "</tr>"

    return html


# 22. FUNÇÃO PARA CRIAR BARRA PRINCIPAL


def criar_barra_principal(titulo):

    return f"""
    <tr>

        <td
            colspan="{len(anos_bal) + 1}"

            style="
                background:{COR_BARRA_PRINCIPAL};
                color:{COR_BRANCO};
                font-weight:bold;
                font-size:14px;
                padding:10px 12px;
                border:1px solid {COR_BARRA_PRINCIPAL};
                text-align:left;
            "
        >
            {titulo}
        </td>

    </tr>
    """


# 23. FUNÇÃO PARA CRIAR SUBSEÇÕES


def criar_secao(titulo):

    return f"""
    <tr>

        <td
            colspan="{len(anos_bal) + 1}"

            style="
                background:{COR_VERDE};
                color:{COR_BRANCO};
                font-weight:bold;
                font-size:14px;
                padding:10px 12px;
                border:1px solid #548235;
                text-align:left;
            "
        >
            {titulo}
        </td>

    </tr>
    """


# 24. FUNÇÃO PARA CRIAR ESPAÇAMENTO


def criar_espaco():

    return f"""
    <tr>

        <td
            colspan="{len(anos_bal) + 1}"

            style="
                height:18px;
                background:{COR_BRANCO};
                border:none;
            "
        ></td>

    </tr>
    """


# 25. CONSTRUÇÃO DA TABELA


linhas_html = ""


# ATIVO

linhas_html += criar_barra_principal("ATIVO")


# ATIVO CIRCULANTE


linhas_html += criar_secao("ATIVO CIRCULANTE")


for conta in ativo_circulante:

    if conta == conta_ativo_circulante:

        linhas_html += criar_linha(
            conta,
            fundo=COR_VERDE_CLARO,
            cor=COR_TEXTO,
            negrito=True
        )

    else:

        linhas_html += criar_linha(
            conta,
            fundo=COR_FUNDO
        )


# ATIVO NÃO CIRCULANTE


linhas_html += criar_secao("ATIVO NÃO CIRCULANTE")


for conta in ativo_nao_circulante:

    if conta in [
        "Ativo Não Circulante",
        "Realizável a Longo Prazo",
        "Permanente"
    ]:

        linhas_html += criar_linha(
            conta,
            fundo=COR_VERDE_CLARO,
            cor=COR_TEXTO,
            negrito=True
        )

    else:

        linhas_html += criar_linha(
            conta,
            fundo=COR_FUNDO
        )



# TOTAL DO ATIVO


linhas_html += criar_linha(
    conta_total_ativo,

    nome_exibido="TOTAL DO ATIVO",

    fundo=COR_TOTAL,
    cor=COR_BRANCO,
    negrito=True,

    borda_superior=f"3px solid {COR_TOTAL_BORDA}",
    borda_inferior=f"3px solid {COR_TOTAL_BORDA}"
)


# ESPAÇO ENTRE ATIVO E PASSIVO


linhas_html += criar_espaco()


# PASSIVO


linhas_html += criar_barra_principal("PASSIVO")


# PASSIVO CIRCULANTE


linhas_html += criar_secao("PASSIVO CIRCULANTE")

for conta in passivo_circulante:

    if conta == conta_passivo_circulante:

        linhas_html += criar_linha(
            conta,
            fundo=COR_VERDE_CLARO,
            cor=COR_TEXTO,
            negrito=True
        )

    else:

        linhas_html += criar_linha(
            conta,
            fundo=COR_FUNDO
        )


# PASSIVO NÃO CIRCULANTE


linhas_html += criar_secao("PASSIVO NÃO CIRCULANTE")

for conta in passivo_nao_circulante:

    if conta in [
        "Passivo Não Circulante",
        "Exigível a Longo Prazo"
    ]:

        linhas_html += criar_linha(
            conta,
            fundo=COR_VERDE_CLARO,
            cor=COR_TEXTO,
            negrito=True
        )

    else:

        linhas_html += criar_linha(
            conta,
            fundo=COR_FUNDO
        )


# TOTAL DO PASSIVO


linhas_html += criar_linha(
    conta_total_passivo,

    nome_exibido="TOTAL DO PASSIVO",

    fundo=COR_TOTAL,
    cor=COR_BRANCO,
    negrito=True,

    borda_superior=f"3px solid {COR_TOTAL_BORDA}",
    borda_inferior=f"3px solid {COR_TOTAL_BORDA}"
)


# ESPAÇO ENTRE PASSIVO E PATRIMÔNIO LÍQUIDO

linhas_html += criar_espaco()

# PATRIMÔNIO LÍQUIDO


linhas_html += criar_barra_principal("PATRIMÔNIO LÍQUIDO")


# CONTAS DO PATRIMÔNIO LÍQUIDO


for conta in patrimonio_detalhes:

    linhas_html += criar_linha(
        conta,
        fundo=COR_FUNDO
    )


# TOTAL DO PATRIMÔNIO LÍQUIDO

linhas_html += criar_linha(
    conta_usada_total_patrimonio,

    nome_exibido="TOTAL DO PATRIMÔNIO LÍQUIDO",

    fundo=COR_TOTAL,
    cor=COR_BRANCO,
    negrito=True,

    borda_superior=f"3px solid {COR_TOTAL_BORDA}",
    borda_inferior=f"3px solid {COR_TOTAL_BORDA}"
)


# 26. TABELA COMPLETA


html_bal = f"""

<div style="
    width:100%;
    overflow-x:auto;
    font-family:Arial, sans-serif;
    margin-top:20px;
    margin-bottom:35px;
">

    <table style="
        border-collapse:collapse;
        width:max-content;
        min-width:100%;
        font-size:13px;
    ">

        <thead>

            <!-- ==================================================
                 TÍTULO PRINCIPAL
                 ================================================== -->

            <tr>

                <th style="
                    background:{COR_TOTAL};
                    color:{COR_BRANCO};
                    padding:11px;
                    text-align:center;
                    border:1px solid {COR_BORDA};
                ">
                    BAL
                </th>


                <th
                    colspan="{len(anos_bal)}"

                    style="
                        background:{COR_TOTAL};
                        color:{COR_BRANCO};
                        padding:11px;
                        text-align:center;
                        border:1px solid {COR_BORDA};
                        font-size:15px;
                    "
                >

                    Balanço Patrimonial
                    — Valores em R$ bilhões

                </th>

            </tr>


            <!-- ==================================================
                 CABEÇALHO DOS ANOS
                 ================================================== -->

            <tr>

                <th style="
                    background:{COR_VERDE};
                    color:{COR_BRANCO};
                    padding:9px 12px;
                    text-align:left;
                    border:1px solid {COR_BORDA};
                    min-width:320px;
                ">

                    CONTA

                </th>

                {cabecalho_anos}

            </tr>

        </thead>


        <tbody>

            {linhas_html}

        </tbody>


    </table>

</div>

"""


# 27. EXIBIÇÃO


display(HTML(html_bal))

In [10]:
import pandas as pd
from IPython.display import display, HTML


# 1. SELEÇÃO DOS DADOS DO FLU


flu = df[df["CONTA"].str.startswith("FLU", na=False)].copy()


# 2. TRATAMENTO DOS VALORES


flu["NOVOVALOR"] = (flu["VALOR"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False))

flu["NOVOVALOR"] = pd.to_numeric(flu["NOVOVALOR"],errors="coerce")


# 3. CONVERSÃO PARA BILHÕES


flu["VALOR_BILHOES"] = (flu["NOVOVALOR"] / 1_000_000_000)


# 4. TRATAMENTO DO ANO


flu["NUM_ANO"] = (flu["ANO"].astype(str).str.extract(r"(\d+)")[0])

flu["NUM_ANO"] = pd.to_numeric(flu["NUM_ANO"],errors="coerce")


# 5. LIMPEZA DOS NOMES DAS CONTAS


flu["CONTA_LIMPA"] = (flu["CONTA"].astype(str).str.replace(r"^FLU\s*-\s*","",regex=True).str.strip())


# 6. SOMA DE TODOS OS CENÁRIOS


flu_consolidado = (flu.groupby(["CONTA_LIMPA", "NUM_ANO"],as_index=False)["VALOR_BILHOES"].sum())


# 7. ORDEM ORIGINAL DAS CONTAS


ordem_contas_flu = (flu["CONTA_LIMPA"].dropna().drop_duplicates().tolist())


# 8. TABELA DINÂMICA


tabela_flu = flu_consolidado.pivot_table(index="CONTA_LIMPA",columns="NUM_ANO",values="VALOR_BILHOES",aggfunc="sum",fill_value=0)

tabela_flu = tabela_flu.reindex(
    [
        conta
        for conta in ordem_contas_flu
        if conta in tabela_flu.index
    ]
)


# Ordena os anos
tabela_flu = tabela_flu.reindex(
    sorted(tabela_flu.columns),
    axis=1
)


# 9. ANOS


anos_flu = [
    int(ano)
    for ano in tabela_flu.columns
]


# 10. CORES PADRÃO


COR_BARRA_PRINCIPAL = "#4472C4"
COR_VERDE = "#70AD47"
COR_VERDE_CLARO = "#C6E0B4"
COR_FUNDO = "#F7FBF4"
COR_TOTAL = "#17365D"
COR_TOTAL_BORDA = "#0E243F"
COR_TEXTO = "#1F2937"
COR_BRANCO = "#FFFFFF"
COR_BORDA = "#D9E2F3"


# 11. FORMATAÇÃO DOS VALORES


def formatar_valor_flu(valor):

    if pd.isna(valor):
        return "R$ -"

    if abs(valor) < 0.005:
        return "R$ -"

    numero = (
        f"{abs(valor):,.2f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    if valor < 0:
        return f"- R$ {numero}"

    return f"R$ {numero}"


# 12. IDENTIFICAÇÃO DAS CONTAS IMPORTANTES


linhas_resultado_final = [
    conta
    for conta in tabela_flu.index
    if conta.lower().strip() == "saldo final"
]

linhas_subtotais = [
    conta
    for conta in tabela_flu.index
    if any(
        palavra in conta.lower()
        for palavra in [
            "geração de caixa",
            "geracao de caixa",
            "resultado financeiro"
        ]
    )
    and conta not in linhas_resultado_final
]

linhas_entrada = [
    conta
    for conta in tabela_flu.index
    if conta.lower().strip() in [
        "saldo inicial",
        "receita",
        "entradas"
    ]
]


# 13. CABEÇALHO DOS ANOS


cabecalho_anos = ""

for ano in anos_flu:

    cabecalho_anos += f"""
    <th style="
        background:{COR_VERDE};
        color:{COR_BRANCO};
        padding:9px 14px;
        text-align:center;
        border:1px solid {COR_BORDA};
        min-width:110px;
    ">
        Ano {ano}
    </th>
    """


# 14. FUNÇÃO PARA CRIAR LINHA


def criar_linha_flu(
    conta,
    fundo=COR_FUNDO,
    cor=COR_TEXTO,
    negrito=False,
    borda_superior=f"1px solid {COR_BORDA}",
    borda_inferior=f"1px solid {COR_BORDA}"
):

    if conta not in tabela_flu.index:
        return ""

    peso = "bold" if negrito else "normal"

    html = f"""
    <tr>

        <td style="
            background:{fundo};
            color:{cor};
            font-weight:{peso};
            padding:9px 12px;
            border-left:1px solid {COR_BORDA};
            border-right:1px solid {COR_BORDA};
            border-top:{borda_superior};
            border-bottom:{borda_inferior};
            white-space:nowrap;
            min-width:320px;
        ">
            {conta}
        </td>
    """

    for ano in anos_flu:

        valor = tabela_flu.loc[
            conta,
            ano
        ]

        html += f"""
        <td style="
            background:{fundo};
            color:{cor};
            font-weight:{peso};
            padding:9px 12px;
            text-align:right;
            border-left:1px solid {COR_BORDA};
            border-right:1px solid {COR_BORDA};
            border-top:{borda_superior};
            border-bottom:{borda_inferior};
            white-space:nowrap;
        ">
            {formatar_valor_flu(valor)}
        </td>
        """

    html += "</tr>"

    return html


# 15. FUNÇÃO PARA CRIAR BARRA PRINCIPAL


def criar_barra_principal_flu(titulo):

    return f"""
    <tr>

        <td
            colspan="{len(anos_flu) + 1}"

            style="
                background:{COR_BARRA_PRINCIPAL};
                color:{COR_BRANCO};
                font-weight:bold;
                font-size:14px;
                padding:10px 12px;
                border:1px solid {COR_BARRA_PRINCIPAL};
                text-align:left;
            "
        >
            {titulo}
        </td>

    </tr>
    """


# 16. CONSTRUÇÃO DAS LINHAS


linhas_html = ""


linhas_html += criar_barra_principal_flu("FLUXO DE CAIXA")


# CONTAS DO FLU


for conta in tabela_flu.index:

    if conta in linhas_resultado_final:

        linhas_html += criar_linha_flu(
            conta,
            fundo=COR_TOTAL,
            cor=COR_BRANCO,
            negrito=True,
            borda_superior=f"3px solid {COR_TOTAL_BORDA}",
            borda_inferior=f"3px solid {COR_TOTAL_BORDA}"
        )

    elif conta in linhas_subtotais:

        linhas_html += criar_linha_flu(
            conta,
            fundo=COR_VERDE,
            cor=COR_BRANCO,
            negrito=True,
            borda_superior="2px solid #548235",
            borda_inferior="2px solid #548235"
        )

    elif conta in linhas_entrada:

        linhas_html += criar_linha_flu(
            conta,
            fundo=COR_VERDE_CLARO,
            cor=COR_TEXTO,
            negrito=True
        )

    else:

        linhas_html += criar_linha_flu(
            conta,
            fundo=COR_FUNDO,
            cor=COR_TEXTO
        )

html_flu = f"""

<div style="
    width:100%;
    overflow-x:auto;
    font-family:Arial, sans-serif;
    margin-top:20px;
    margin-bottom:35px;
">

    <table style="
        border-collapse:collapse;
        width:max-content;
        min-width:100%;
        font-size:13px;
    ">

        <thead>


            <!-- ==================================================
                 TÍTULO PRINCIPAL
                 ================================================== -->

            <tr>

                <th style="
                    background:{COR_TOTAL};
                    color:{COR_BRANCO};
                    padding:11px;
                    text-align:center;
                    border:1px solid {COR_BORDA};
                ">
                    FLU
                </th>


                <th
                    colspan="{len(anos_flu)}"

                    style="
                        background:{COR_TOTAL};
                        color:{COR_BRANCO};
                        padding:11px;
                        text-align:center;
                        border:1px solid {COR_BORDA};
                        font-size:15px;
                    "
                >

                    Fluxo de Caixa
                    — Valores em R$ bilhões

                </th>

            </tr>


            <!-- ==================================================
                 CABEÇALHO DOS ANOS
                 ================================================== -->

            <tr>

                <th style="
                    background:{COR_VERDE};
                    color:{COR_BRANCO};
                    padding:9px 12px;
                    text-align:left;
                    border:1px solid {COR_BORDA};
                    min-width:320px;
                ">

                    CONTA

                </th>

                {cabecalho_anos}

            </tr>

        </thead>


        <tbody>

            {linhas_html}

        </tbody>


    </table>

</div>

"""


# 18. EXIBIÇÃO


display(HTML(html_flu))

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets


# 1. CARREGAMENTO DA BASE


arquivo = "/content/Demonstrativo.csv"

df = pd.read_csv(arquivo,sep=";",header=None,encoding="latin1")

df.columns = ["ANO","CENARIO","CONTA","VALOR"]


# 2. CORREÇÃO DE TEXTO


def corrigir_texto(texto):

    if pd.isna(texto):
        return texto

    try:
        return str(texto).encode("latin1").decode("utf-8")
    except:
        return texto


df["CONTA"] = df["CONTA"].apply(corrigir_texto)


# 3. LIMPEZA DOS VALORES


df["VALOR_NUMERICO"] = (
    df["VALOR"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["VALOR_NUMERICO"] = pd.to_numeric(
    df["VALOR_NUMERICO"],
    errors="coerce"
)


# 4. ANO NUMÉRICO


df["NUM_ANO"] = (
    df["ANO"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
)

df["NUM_ANO"] = pd.to_numeric(
    df["NUM_ANO"],
    errors="coerce"
)


# 5. CENÁRIO NUMÉRICO


df["NUM_CENARIO"] = (
    df["CENARIO"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
)

df["NUM_CENARIO"] = pd.to_numeric(
    df["NUM_CENARIO"],
    errors="coerce"
)


# 6. VALOR EM BILHÕES


df["VALOR_BILHOES"] = (
    df["VALOR_NUMERICO"]
    / 1_000_000_000
)


# 7. IDENTIFICAÇÃO DO DEMONSTRATIVO


def identificar_demonstrativo(conta):

    if pd.isna(conta):
        return "OUTROS"

    conta = str(conta).strip()

    if conta.startswith("BAL"):
        return "BAL"

    elif conta.startswith("DRE"):
        return "DRE"

    elif conta.startswith("FLU"):
        return "FLU"

    else:
        return "OUTROS"


df["DEMONSTRATIVO"] = (df["CONTA"].apply(identificar_demonstrativo))


# 8. LIMPEZA DO NOME DAS CONTAS


df["CONTA_LIMPA"] = (df["CONTA"].astype(str).str.replace(r"^(BAL|DRE|FLU)\s*-\s*","",regex=True).str.strip())


# 9. FILTRA SOMENTE BAL / DRE / FLU

dados = df[
    df["DEMONSTRATIVO"].isin(
        [
            "BAL",
            "DRE",
            "FLU"
        ]
    )
].copy()


# 10. CONSOLIDA OS CENÁRIOS

dados_integrados = (
    dados
    .groupby(
        [
            "DEMONSTRATIVO",
            "CONTA_LIMPA",
            "NUM_ANO"
        ],
        as_index=False
    )["VALOR_BILHOES"]
    .sum()
)


# 11. CORES

COR_AZUL_ESCURO = "#17365D"
COR_AZUL = "#4472C4"
COR_VERDE = "#70AD47"
COR_VERDE_CLARO = "#C6E0B4"
COR_FUNDO = "#F7FBF4"
COR_TEXTO = "#1F2937"


# 12. FORMATAÇÃO DOS VALORES

def formatar_valor(valor):

    if pd.isna(valor):
        return "R$ -"

    if abs(valor) < 0.005:
        return "R$ -"

    numero = (
        f"{abs(valor):,.2f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    if valor < 0:
        return f"- R$ {numero}"

    return f"R$ {numero}"


# 13. WIDGET - DEMONSTRATIVO

seletor_demonstrativo = widgets.Dropdown(

    options=[
        "BAL",
        "DRE",
        "FLU"
    ],

    value="BAL",

    description="Demonstrativo:",

    style={
        "description_width": "110px"
    },

    layout=widgets.Layout(
        width="300px"
    )
)


# 14. WIDGET - CONTA

seletor_conta = widgets.Dropdown(

    options=["TODAS"],

    value="TODAS",

    description="Conta:",

    style={
        "description_width": "110px"
    },

    layout=widgets.Layout(
        width="500px"
    )
)


# 15. WIDGET - ANO INICIAL

anos_disponiveis = sorted(
    dados_integrados["NUM_ANO"]
    .dropna()
    .astype(int)
    .unique()
)

seletor_ano_inicio = widgets.Dropdown(

    options=anos_disponiveis,

    value=min(anos_disponiveis),

    description="Ano inicial:",

    style={
        "description_width": "110px"
    },

    layout=widgets.Layout(
        width="250px"
    )
)


# 16. WIDGET - ANO FINAL

seletor_ano_fim = widgets.Dropdown(

    options=anos_disponiveis,

    value=max(anos_disponiveis),

    description="Ano final:",

    style={
        "description_width": "110px"
    },

    layout=widgets.Layout(
        width="250px"
    )
)


# 17. WIDGET - TIPO DE VISUALIZAÇÃO

seletor_visualizacao = widgets.ToggleButtons(

    options=[
        "Tabela",
        "Linha",
        "Barras"
    ],

    value="Tabela",

    description="Visualização:",

    style={
        "description_width": "110px"
    }
)


# 18. WIDGET - TOP N CONTAS

seletor_top = widgets.Dropdown(

    options=[
        ("Todas", 0),
        ("Top 5", 5),
        ("Top 10", 10),
        ("Top 15", 15)
    ],

    value=0,

    description="Quantidade:",

    style={
        "description_width": "110px"
    },

    layout=widgets.Layout(
        width="250px"
    )
)


# 19. ÁREA DE SAÍDA

saida = widgets.Output()


# 20. ATUALIZA LISTA DE CONTAS

def atualizar_contas(*args):

    demonstrativo = seletor_demonstrativo.value

    contas = (
        dados_integrados[
            dados_integrados["DEMONSTRATIVO"]
            == demonstrativo
        ]["CONTA_LIMPA"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    seletor_conta.options = (
        ["TODAS"]
        + contas
    )

    seletor_conta.value = "TODAS"


# 21. CRIAÇÃO DA TABELA HTML

def criar_tabela_html(tabela, titulo):

    anos = tabela.columns.tolist()

    cabecalho = ""

    for ano in anos:

        cabecalho += f"""
        <th style="
            background:{COR_VERDE};
            color:white;
            padding:10px 14px;
            border:1px solid #D9E2F3;
            text-align:center;
        ">
            Ano {int(ano)}
        </th>
        """

    linhas = ""

    for conta in tabela.index:

        linhas += f"""
        <tr>

            <td style="
                background:{COR_FUNDO};
                padding:9px 12px;
                border:1px solid #D9E2F3;
                min-width:300px;
                color:{COR_TEXTO};
            ">
                {conta}
            </td>
        """

        for ano in anos:

            valor = tabela.loc[
                conta,
                ano
            ]

            linhas += f"""
            <td style="
                background:{COR_FUNDO};
                padding:9px 12px;
                border:1px solid #D9E2F3;
                text-align:right;
                min-width:110px;
                color:{COR_TEXTO};
            ">
                {formatar_valor(valor)}
            </td>
            """

        linhas += "</tr>"

    html = f"""
    <div style="
        overflow-x:auto;
        width:100%;
        font-family:Arial;
        margin-top:15px;
    ">

        <table style="
            border-collapse:collapse;
            min-width:100%;
            width:max-content;
            font-size:13px;
        ">

            <thead>

                <tr>

                    <th
                        colspan="{len(anos) + 1}"

                        style="
                            background:{COR_AZUL};
                            color:white;
                            padding:12px;
                            text-align:left;
                            font-size:16px;
                            border:1px solid #D9E2F3;
                        "
                    >
                        {titulo}
                    </th>

                </tr>


                <tr>

                    <th style="
                        background:{COR_VERDE};
                        color:white;
                        padding:10px 12px;
                        border:1px solid #D9E2F3;
                        text-align:left;
                    ">
                        CONTA
                    </th>

                    {cabecalho}

                </tr>

            </thead>

            <tbody>

                {linhas}

            </tbody>

        </table>

    </div>
    """

    return html


# 22. FUNÇÃO PRINCIPAL DO DASHBOARD


def atualizar_dashboard(*args):

    with saida:

        clear_output(wait=True)

        demonstrativo = (
            seletor_demonstrativo.value
        )

        conta = (
            seletor_conta.value
        )

        ano_inicio = (
            seletor_ano_inicio.value
        )

        ano_fim = (
            seletor_ano_fim.value
        )

        visualizacao = (
            seletor_visualizacao.value
        )

        top_n = (
            seletor_top.value
        )

        if ano_inicio > ano_fim:

            display(
                HTML(
                    """
                    <div style="
                        padding:15px;
                        background:#F8D7DA;
                        color:#842029;
                        border-radius:6px;
                        font-family:Arial;
                    ">
                        O ano inicial não pode ser maior
                        que o ano final.
                    </div>
                    """
                )
            )

            return

        filtro = dados_integrados[
            (
                dados_integrados["DEMONSTRATIVO"]
                == demonstrativo
            )
            &
            (
                dados_integrados["NUM_ANO"]
                >= ano_inicio
            )
            &
            (
                dados_integrados["NUM_ANO"]
                <= ano_fim
            )
        ].copy()

        if conta != "TODAS":

            filtro = filtro[
                filtro["CONTA_LIMPA"]
                == conta
            ]

        if top_n > 0 and conta == "TODAS":

            ranking = (
                filtro
                .groupby("CONTA_LIMPA")[
                    "VALOR_BILHOES"
                ]
                .apply(
                    lambda x:
                    x.abs().sum()
                )
                .sort_values(
                    ascending=False
                )
                .head(top_n)
                .index
            )

            filtro = filtro[
                filtro["CONTA_LIMPA"]
                .isin(ranking)
            ]

        if filtro.empty:

            display(
                HTML(
                    """
                    <div style="
                        background:#FFF3CD;
                        color:#664D03;
                        padding:15px;
                        border-radius:6px;
                        font-family:Arial;
                    ">
                        Nenhum dado encontrado
                        para os filtros selecionados.
                    </div>
                    """
                )
            )

            return

        # Tabela dinâmica

        tabela = filtro.pivot_table(index="CONTA_LIMPA",columns="NUM_ANO",values="VALOR_BILHOES",aggfunc="sum",
            fill_value=0
        )


        tabela = tabela.reindex(
            sorted(tabela.columns),
            axis=1
        )

        valor_total = filtro[
            "VALOR_BILHOES"
        ].sum()

        maior_valor = filtro[
            "VALOR_BILHOES"
        ].max()

        menor_valor = filtro[
            "VALOR_BILHOES"
        ].min()

        quantidade_contas = filtro[
            "CONTA_LIMPA"
        ].nunique()


        indicadores = f"""
        <div style="
            display:flex;
            gap:12px;
            flex-wrap:wrap;
            margin-top:15px;
            margin-bottom:20px;
            font-family:Arial;
        ">

            <div style="
                background:{COR_AZUL_ESCURO};
                color:white;
                padding:15px 20px;
                border-radius:8px;
                min-width:180px;
            ">
                <b>Demonstrativo</b><br>
                {demonstrativo}
            </div>

            <div style="
                background:{COR_AZUL};
                color:white;
                padding:15px 20px;
                border-radius:8px;
                min-width:180px;
            ">
                <b>Contas</b><br>
                {quantidade_contas}
            </div>

            <div style="
                background:{COR_VERDE};
                color:white;
                padding:15px 20px;
                border-radius:8px;
                min-width:180px;
            ">
                <b>Valor total</b><br>
                {formatar_valor(valor_total)}
            </div>

            <div style="
                background:{COR_VERDE_CLARO};
                color:{COR_TEXTO};
                padding:15px 20px;
                border-radius:8px;
                min-width:180px;
            ">
                <b>Período</b><br>
                Ano {ano_inicio} até Ano {ano_fim}
            </div>

        </div>
        """

        display(
            HTML(indicadores)
        )

        # TABELA

        if visualizacao == "Tabela":

            titulo = (
                f"{demonstrativo} — "
                f"Valores em R$ bilhões"
            )

            display(
                HTML(
                    criar_tabela_html(
                        tabela,
                        titulo
                    )
                )
            )

        # GRÁFICO DE LINHAS

        elif visualizacao == "Linha":

            plt.figure(
                figsize=(14, 7)
            )

            for nome_conta in tabela.index:

                plt.plot(
                    tabela.columns,
                    tabela.loc[
                        nome_conta
                    ].values,
                    marker="o",
                    label=nome_conta
                )

            plt.title(
                f"{demonstrativo} - Evolução por Ano"
            )

            plt.xlabel(
                "Ano"
            )

            plt.ylabel(
                "R$ bilhões"
            )

            plt.grid(
                alpha=0.3
            )

            plt.legend(
                bbox_to_anchor=(
                    1.05,
                    1
                ),
                loc="upper left"
            )

            plt.tight_layout()

            plt.show()

        # GRÁFICO DE BARRAS

        elif visualizacao == "Barras":

            tabela.T.plot(
                kind="bar",
                figsize=(15, 7)
            )

            plt.title(
                f"{demonstrativo} - Comparação por Ano"
            )

            plt.xlabel(
                "Ano"
            )

            plt.ylabel(
                "R$ bilhões"
            )

            plt.xticks(
                rotation=0
            )

            plt.grid(
                axis="y",
                alpha=0.3
            )

            plt.legend(
                bbox_to_anchor=(
                    1.05,
                    1
                ),
                loc="upper left"
            )

            plt.tight_layout()

            plt.show()



# 23. EVENTOS DOS FILTROS


seletor_demonstrativo.observe(
    atualizar_contas,
    names="value"
)

seletor_demonstrativo.observe(
    atualizar_dashboard,
    names="value"
)

seletor_conta.observe(
    atualizar_dashboard,
    names="value"
)

seletor_ano_inicio.observe(
    atualizar_dashboard,
    names="value"
)

seletor_ano_fim.observe(
    atualizar_dashboard,
    names="value"
)

seletor_visualizacao.observe(
    atualizar_dashboard,
    names="value"
)

seletor_top.observe(
    atualizar_dashboard,
    names="value"
)



# 24. INICIALIZAÇÃO


atualizar_contas()



# 25. TÍTULO DO DASHBOARD


display(
    HTML(
        f"""
        <div style="
            background:{COR_AZUL_ESCURO};
            color:white;
            padding:18px 20px;
            border-radius:8px;
            margin-bottom:20px;
            font-family:Arial;
        ">

            <div style="
                font-size:24px;
                font-weight:bold;
            ">
                Dashboard Financeiro
            </div>

            <div style="
                font-size:14px;
                margin-top:5px;
            ">
                BAL • DRE • FLU
                — Valores em R$ bilhões
            </div>

        </div>
        """
    )
)



# 26. PAINEL DE FILTROS


linha_1 = widgets.HBox(
    [
        seletor_demonstrativo,
        seletor_conta
    ]
)

linha_2 = widgets.HBox(
    [
        seletor_ano_inicio,
        seletor_ano_fim,
        seletor_top
    ]
)

linha_3 = widgets.HBox(
    [
        seletor_visualizacao
    ]
)


painel_filtros = widgets.VBox(
    [
        linha_1,
        linha_2,
        linha_3
    ]
)


display(
    HTML(
        """
        <div style="
            font-family:Arial;
            font-size:16px;
            font-weight:bold;
            margin-bottom:10px;
        ">
            Filtros
        </div>
        """
    )
)


display(
    painel_filtros
)


# 27. RESULTADO

display(
    saida
)


# Primeira atualização
atualizar_dashboard()

Output()